# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [ ]:
%pip install iso3166

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [ ]:
%pip install --upgrade plotly

Requirement already up-to-date: plotly in /usr/local/lib/python3.6/dist-packages (4.12.0)


### Import Statements

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

/usr/local/lib/python3.6/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning:

pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.



### Notebook Presentation

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [ ]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [ ]:
# Shape of the dataframe
print(f'Shape: {df_data.shape}')
print(f'Rows: {df_data.shape[0]}, Columns: {df_data.shape[1]}')
print(f'\nColumn Names: {df_data.columns.tolist()}')
df_data.head()

In [ ]:
# Check data types and non-null counts
df_data.info()
print(f'\nAny NaN values? {df_data.isnull().values.any()}')
print(f'\nNaN per column:\n{df_data.isnull().sum()}')
print(f'\nNumber of duplicates: {df_data.duplicated().sum()}')

## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data. 

In [ ]:
# Drop the junk index columns
df_data.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)
print(f'Columns after cleanup: {df_data.columns.tolist()}')
print(f'\nRemaining NaN values per column:\n{df_data.isnull().sum()}')

In [ ]:
# Convert Price to numeric (it's stored as string)
df_data['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')

# Convert Date to datetime
df_data['Date'] = pd.to_datetime(df_data['Date'], utc=True)

print('Data types after conversion:')
print(df_data.dtypes)
df_data.head()

## Descriptive Statistics

In [ ]:
# Descriptive statistics for numeric columns
df_data.describe()

In [ ]:
# Descriptive statistics for categorical columns
print('Unique Organisations:', df_data['Organisation'].nunique())
print('Unique Locations:', df_data['Location'].nunique())
print('Unique Rockets (Detail):', df_data['Detail'].nunique())
print(f'\nRocket Status:\n{df_data["Rocket_Status"].value_counts()}')
print(f'\nMission Status:\n{df_data["Mission_Status"].value_counts()}')
print(f'\nPrice range: {df_data["Price"].min()} - {df_data["Price"].max()} (USD millions)')
print(f'Date range: {df_data["Date"].min()} to {df_data["Date"].max()}')

# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [ ]:
# Number of launches per organisation
launches_per_org = df_data['Organisation'].value_counts()
print(launches_per_org)

In [ ]:
# Bar chart of launches per organisation (top 20)
top_orgs = launches_per_org.head(20)

fig = px.bar(
    x=top_orgs.values,
    y=top_orgs.index,
    orientation='h',
    title='Number of Space Mission Launches by Organisation (Top 20)',
    labels={'x': 'Number of Launches', 'y': 'Organisation'},
    color=top_orgs.values,
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [ ]:
# Count of active vs retired rockets
rocket_status_counts = df_data['Rocket_Status'].value_counts()
print(rocket_status_counts)

In [ ]:
# Pie chart: Active vs Retired Rockets
labels = rocket_status_counts.index.str.replace('Status', '')

fig = px.pie(
    values=rocket_status_counts.values,
    names=labels,
    title='Active vs Retired Rockets',
    color_discrete_sequence=['#2ecc71', '#e74c3c'],
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.show()

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [ ]:
# Count of each mission status
mission_status_counts = df_data['Mission_Status'].value_counts()
print(mission_status_counts)

In [ ]:
# Pie chart of mission status distribution
fig = px.pie(
    values=mission_status_counts.values,
    names=mission_status_counts.index,
    title='Distribution of Mission Status',
    color=mission_status_counts.index,
    color_discrete_map={
        'Success': '#2ecc71',
        'Failure': '#e74c3c',
        'Partial Failure': '#f39c12',
        'Prelaunch Failure': '#9b59b6'
    },
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.show()

# How Expensive are the Launches? 

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values). 

In [ ]:
# Price distribution - basic stats
print(f'Number of missions with price data: {df_data["Price"].notna().sum()}')
print(f'Number of missions without price data: {df_data["Price"].isna().sum()}')
print(f'\nPrice statistics (USD millions):')
df_data['Price'].describe()

In [ ]:
# Histogram of launch prices
fig = px.histogram(
    df_data,
    x='Price',
    nbins=50,
    title='Distribution of Launch Prices (USD Millions)',
    labels={'Price': 'Price (USD Millions)', 'count': 'Number of Launches'},
    color_discrete_sequence=['#3498db'],
    marginal='box'
)
fig.update_layout(bargap=0.05)
fig.show()

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map. 
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [ ]:
# Extract country from the Location column (last part after comma)
df_data['Country'] = df_data['Location'].apply(lambda x: x.split(', ')[-1].strip())

# Fix country names that no longer exist or are special locations
country_mapping = {
    'Russia': 'Russian Federation',
    'New Mexico': 'United States',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran, Islamic Republic of',
    'Pacific Missile Range Facility': 'United States',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'United States',
    'Kenya': 'Kenya',
    'USA': 'United States',
    'North Korea': "Korea, Democratic People's Republic of",
    'South Korea': 'Korea, Republic of',
    'Iran': 'Iran, Islamic Republic of'
}

df_data['Country'] = df_data['Country'].replace(country_mapping)
print('Unique countries:', df_data['Country'].nunique())
print(df_data['Country'].value_counts())

In [ ]:
# Convert country names to ISO Alpha-3 codes
def get_alpha3(country_name):
    try:
        return countries.get(country_name).alpha3
    except:
        return None

launches_by_country = df_data['Country'].value_counts().reset_index()
launches_by_country.columns = ['Country', 'Launches']
launches_by_country['Alpha3'] = launches_by_country['Country'].apply(get_alpha3)

fig = px.choropleth(
    launches_by_country,
    locations='Alpha3',
    color='Launches',
    hover_name='Country',
    title='Total Number of Space Mission Launches by Country',
    color_continuous_scale='matter'
)
fig.update_layout(height=600)
fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [ ]:
# Filter only failures
failures = df_data[df_data['Mission_Status'].isin(['Failure', 'Partial Failure', 'Prelaunch Failure'])]
failures_by_country = failures['Country'].value_counts().reset_index()
failures_by_country.columns = ['Country', 'Failures']
failures_by_country['Alpha3'] = failures_by_country['Country'].apply(get_alpha3)
print(failures_by_country)

In [ ]:
# Choropleth map of failures
fig = px.choropleth(
    failures_by_country,
    locations='Alpha3',
    color='Failures',
    hover_name='Country',
    title='Number of Space Mission Failures by Country',
    color_continuous_scale='Reds'
)
fig.update_layout(height=600)
fig.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status. 

In [ ]:
# Prepare data for sunburst chart
sunburst_data = df_data.groupby(['Country', 'Organisation', 'Mission_Status']).size().reset_index(name='Count')
sunburst_data.head()

In [ ]:
# Create Sunburst chart
fig = px.sunburst(
    sunburst_data,
    path=['Country', 'Organisation', 'Mission_Status'],
    values='Count',
    title='Space Missions: Country > Organisation > Mission Status',
    color='Count',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Alternative: Sunburst with only top countries for clarity
top_countries = df_data['Country'].value_counts().head(10).index.tolist()
sunburst_top = sunburst_data[sunburst_data['Country'].isin(top_countries)]

fig = px.sunburst(
    sunburst_top,
    path=['Country', 'Organisation', 'Mission_Status'],
    values='Count',
    title='Space Missions by Top 10 Countries: Country > Organisation > Status',
)
fig.update_layout(height=700)
fig.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [ ]:
# Total spending by organisation (only missions with price data)
spending_by_org = df_data.groupby('Organisation')['Price'].sum().dropna().sort_values(ascending=False)
print(spending_by_org.head(20))

In [ ]:
# Bar chart of total spending by organisation (top 20)
top_spending = spending_by_org.head(20)

fig = px.bar(
    x=top_spending.values,
    y=top_spending.index,
    orientation='h',
    title='Total Money Spent on Space Missions by Organisation (Top 20, USD Millions)',
    labels={'x': 'Total Spending (USD Millions)', 'y': 'Organisation'},
    color=top_spending.values,
    color_continuous_scale='Plasma'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

In [ ]:
# Number of launches vs total spending
org_stats = df_data.groupby('Organisation').agg(
    total_launches=('Organisation', 'count'),
    total_spending=('Price', 'sum')
).dropna().sort_values('total_spending', ascending=False).head(20)

fig = px.scatter(
    org_stats,
    x='total_launches',
    y='total_spending',
    text=org_stats.index,
    title='Total Launches vs Total Spending by Organisation',
    labels={'total_launches': 'Number of Launches', 'total_spending': 'Total Spending (USD Millions)'},
    size='total_spending',
    color='total_spending',
    color_continuous_scale='Viridis'
)
fig.update_traces(textposition='top center')
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

In [ ]:
# Average cost per launch by organisation
cost_per_launch = df_data.groupby('Organisation')['Price'].mean().dropna().sort_values(ascending=False)
print(cost_per_launch.head(20))

In [ ]:
# Bar chart: average cost per launch (top 20)
top_cost = cost_per_launch.head(20)

fig = px.bar(
    x=top_cost.values,
    y=top_cost.index,
    orientation='h',
    title='Average Cost per Launch by Organisation (Top 20, USD Millions)',
    labels={'x': 'Average Cost per Launch (USD Millions)', 'y': 'Organisation'},
    color=top_cost.values,
    color_continuous_scale='Sunset'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

In [ ]:
# Box plot: price distribution per top organisation
top_org_names = df_data['Organisation'].value_counts().head(10).index.tolist()
df_top_orgs = df_data[df_data['Organisation'].isin(top_org_names)]

fig = px.box(
    df_top_orgs,
    x='Organisation',
    y='Price',
    title='Launch Price Distribution for Top 10 Organisations',
    labels={'Price': 'Price (USD Millions)'},
    color='Organisation'
)
fig.update_layout(showlegend=False, xaxis_tickangle=-45)
fig.show()

# Chart the Number of Launches per Year

In [ ]:
# Extract year from Date
df_data['Year'] = df_data['Date'].dt.year
launches_per_year = df_data['Year'].value_counts().sort_index()
print(launches_per_year)

In [ ]:
# Line chart of launches per year
fig = px.line(
    x=launches_per_year.index,
    y=launches_per_year.values,
    title='Number of Space Mission Launches per Year',
    labels={'x': 'Year', 'y': 'Number of Launches'},
    markers=True
)
fig.update_layout(xaxis=dict(dtick=5))
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [ ]:
# Monthly launches with rolling average
df_data['YearMonth'] = df_data['Date'].dt.to_period('M')
monthly_launches = df_data.groupby('YearMonth').size()
monthly_launches.index = monthly_launches.index.to_timestamp()

# Rolling 12-month average
rolling_avg = monthly_launches.rolling(window=12).mean()

print(f'Month with highest launches: {monthly_launches.idxmax()} ({monthly_launches.max()} launches)')

In [ ]:
# Plot month-on-month launches with rolling average
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(monthly_launches.index, monthly_launches.values, alpha=0.4, label='Monthly Launches', color='#3498db')
ax.plot(rolling_avg.index, rolling_avg.values, linewidth=2.5, label='12-Month Rolling Average', color='#e74c3c')
ax.set_title('Number of Launches Month-on-Month', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Launches')
ax.legend()
plt.tight_layout()
plt.show()

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [ ]:
# Launches by month of the year
df_data['Month'] = df_data['Date'].dt.month
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
launches_by_month = df_data['Month'].value_counts().sort_index()
launches_by_month.index = [month_names[m-1] for m in launches_by_month.index]
print(launches_by_month)

In [ ]:
# Bar chart of launches by month
fig = px.bar(
    x=launches_by_month.index,
    y=launches_by_month.values,
    title='Number of Launches per Month (All Time)',
    labels={'x': 'Month', 'y': 'Number of Launches'},
    color=launches_by_month.values,
    color_continuous_scale='Bluered'
)
fig.update_layout(showlegend=False)
fig.show()

# How has the Launch Price varied Over Time? 

Create a line chart that shows the average price of rocket launches over time. 

In [ ]:
# Average launch price per year
avg_price_per_year = df_data.groupby('Year')['Price'].mean().dropna()
print(avg_price_per_year)

In [ ]:
# Line chart of average price over time
fig = px.line(
    x=avg_price_per_year.index,
    y=avg_price_per_year.values,
    title='Average Launch Price Over Time (USD Millions)',
    labels={'x': 'Year', 'y': 'Average Price (USD Millions)'},
    markers=True
)
fig.update_traces(line_color='#e74c3c')
fig.update_layout(xaxis=dict(dtick=5))
fig.show()

# Chart the Number of Launches over Time by the Top 10 Organisations. 

How has the dominance of launches changed over time between the different players? 

In [ ]:
# Get top 10 organisations by total launches
top10_orgs = df_data['Organisation'].value_counts().head(10).index.tolist()
print('Top 10 Organisations:', top10_orgs)

In [ ]:
# Launches per year by top 10 organisations
df_top10 = df_data[df_data['Organisation'].isin(top10_orgs)]
launches_top10_year = df_top10.groupby(['Year', 'Organisation']).size().reset_index(name='Launches')
launches_top10_year.head()

In [ ]:
# Area chart: launches over time by top 10 organisations
fig = px.area(
    launches_top10_year,
    x='Year',
    y='Launches',
    color='Organisation',
    title='Number of Launches Over Time by Top 10 Organisations',
    labels={'Launches': 'Number of Launches'},
)
fig.update_layout(xaxis=dict(dtick=5), height=600)
fig.show()

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

In [ ]:
# Filter for Cold War era (up to 1991)
cold_war = df_data[df_data['Year'] <= 1991].copy()

# Define USA and USSR related countries
usa_countries = ['United States']
ussr_countries = ['Russian Federation', 'Kazakhstan']

cold_war['Superpower'] = cold_war['Country'].apply(
    lambda x: 'USA' if x in usa_countries else ('USSR' if x in ussr_countries else 'Other')
)

cold_war_super = cold_war[cold_war['Superpower'].isin(['USA', 'USSR'])]
print(cold_war_super['Superpower'].value_counts())

In [ ]:
# Launches per year during Cold War: USA vs USSR
cold_war_yearly = cold_war_super.groupby(['Year', 'Superpower']).size().reset_index(name='Launches')

fig = px.line(
    cold_war_yearly,
    x='Year',
    y='Launches',
    color='Superpower',
    title='Cold War Space Race: USA vs USSR (Launches per Year)',
    labels={'Launches': 'Number of Launches'},
    markers=True,
    color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
)
fig.update_layout(xaxis=dict(dtick=2))
fig.show()

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches. 

In [ ]:
# Total launches: USA vs USSR during Cold War
super_totals = cold_war_super['Superpower'].value_counts()
print(super_totals)

In [ ]:
# Pie chart: USA vs USSR total launches
fig = px.pie(
    values=super_totals.values,
    names=super_totals.index,
    title='Total Space Mission Launches: USA vs USSR (Cold War Era)',
    color=super_totals.index,
    color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'},
    hole=0.3
)
fig.update_traces(textinfo='percent+label+value')
fig.show()

## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [ ]:
# Cumulative launches year-on-year: USA vs USSR
cumulative = cold_war_yearly.copy()
cumulative['Cumulative'] = cumulative.groupby('Superpower')['Launches'].cumsum()
cumulative.tail()

In [ ]:
# Line chart of cumulative launches
fig = px.line(
    cumulative,
    x='Year',
    y='Cumulative',
    color='Superpower',
    title='Cumulative Number of Launches: USA vs USSR',
    labels={'Cumulative': 'Total Launches (Cumulative)'},
    markers=True,
    color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
)
fig.update_layout(xaxis=dict(dtick=2))
fig.show()

## Chart the Total Number of Mission Failures Year on Year.

In [ ]:
# Mission failures per year for USA and USSR
cold_war_failures = cold_war_super[cold_war_super['Mission_Status'] != 'Success']
failures_yearly = cold_war_failures.groupby(['Year', 'Superpower']).size().reset_index(name='Failures')
print(failures_yearly.head(10))

In [ ]:
# Line chart of failures year-on-year
fig = px.line(
    failures_yearly,
    x='Year',
    y='Failures',
    color='Superpower',
    title='Total Mission Failures Year-on-Year: USA vs USSR',
    labels={'Failures': 'Number of Failures'},
    markers=True,
    color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
)
fig.update_layout(xaxis=dict(dtick=2))
fig.show()

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [ ]:
# Calculate failure percentage per year per superpower
total_per_year = cold_war_super.groupby(['Year', 'Superpower']).size().reset_index(name='Total')
failures_per_year = cold_war_failures.groupby(['Year', 'Superpower']).size().reset_index(name='Failures')

failure_pct = total_per_year.merge(failures_per_year, on=['Year', 'Superpower'], how='left')
failure_pct['Failures'] = failure_pct['Failures'].fillna(0)
failure_pct['Failure_Pct'] = (failure_pct['Failures'] / failure_pct['Total']) * 100
failure_pct.head(10)

In [ ]:
# Line chart of failure percentage over time
fig = px.line(
    failure_pct,
    x='Year',
    y='Failure_Pct',
    color='Superpower',
    title='Percentage of Mission Failures Over Time: USA vs USSR',
    labels={'Failure_Pct': 'Failure Percentage (%)'},
    markers=True,
    color_discrete_map={'USA': '#3498db', 'USSR': '#e74c3c'}
)
fig.update_layout(xaxis=dict(dtick=2), yaxis=dict(range=[0, 100]))
fig.show()

In [ ]:
# Overall failure percentage over time (all countries)
all_yearly = df_data.groupby('Year').size().reset_index(name='Total')
all_failures_yearly = df_data[df_data['Mission_Status'] != 'Success'].groupby('Year').size().reset_index(name='Failures')
all_pct = all_yearly.merge(all_failures_yearly, on='Year', how='left')
all_pct['Failures'] = all_pct['Failures'].fillna(0)
all_pct['Failure_Pct'] = (all_pct['Failures'] / all_pct['Total']) * 100

fig = px.line(
    all_pct,
    x='Year',
    y='Failure_Pct',
    title='Overall Mission Failure Percentage Over Time (All Countries)',
    labels={'Failure_Pct': 'Failure Percentage (%)'},
    markers=True
)
fig.update_traces(line_color='#e74c3c')
fig.update_layout(xaxis=dict(dtick=5))
fig.show()

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches? 

In [ ]:
# For each year, find the country with the most launches
launches_country_year = df_data.groupby(['Year', 'Country']).size().reset_index(name='Launches')
idx = launches_country_year.groupby('Year')['Launches'].idxmax()
leading_country = launches_country_year.loc[idx][['Year', 'Country', 'Launches']]
print('Leading country by total launches per year:')
print(leading_country.to_string(index=False))

In [ ]:
# Same analysis but only successful launches
successful = df_data[df_data['Mission_Status'] == 'Success']
success_country_year = successful.groupby(['Year', 'Country']).size().reset_index(name='Launches')
idx_s = success_country_year.groupby('Year')['Launches'].idxmax()
leading_success = success_country_year.loc[idx_s][['Year', 'Country', 'Launches']]

print('Leading country by successful launches per year:')
print(leading_success.to_string(index=False))

# Compare
comparison = leading_country.merge(leading_success, on='Year', suffixes=('_Total', '_Successful'))
diff = comparison[comparison['Country_Total'] != comparison['Country_Successful']]
print(f'\nYears where leader changes when counting only successes: {len(diff)}')
if len(diff) > 0:
    print(diff.to_string(index=False))

# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020? 

In [ ]:
# For each year, find the organisation with the most launches
launches_org_year = df_data.groupby(['Year', 'Organisation']).size().reset_index(name='Launches')
idx_org = launches_org_year.groupby('Year')['Launches'].idxmax()
leading_org = launches_org_year.loc[idx_org][['Year', 'Organisation', 'Launches']]
print('Leading organisation by launches per year:')
print(leading_org.to_string(index=False))

In [ ]:
# Bar chart showing leading organisation per year
fig = px.bar(
    leading_org,
    x='Year',
    y='Launches',
    color='Organisation',
    title='Organisation with Most Launches per Year',
    labels={'Launches': 'Number of Launches'},
)
fig.update_layout(xaxis=dict(dtick=2), height=600)
fig.show()

In [ ]:
# Summary: Which organisation dominated in specific decades?
print('=== 1970s dominant org ===')
seventies = df_data[(df_data['Year'] >= 1970) & (df_data['Year'] < 1980)]
print(seventies['Organisation'].value_counts().head(3))

print('\n=== 1980s dominant org ===')
eighties = df_data[(df_data['Year'] >= 1980) & (df_data['Year'] < 1990)]
print(eighties['Organisation'].value_counts().head(3))

print('\n=== 2018-2020 dominant org ===')
recent = df_data[df_data['Year'].isin([2018, 2019, 2020])]
print(recent['Organisation'].value_counts().head(3))